# Importing system files

### Make sure you are using the same python version as the env

In [12]:
import sys, platform
print(sys.executable)
print(platform.python_version())

/opt/miniconda3/envs/geospatial_env/bin/python
3.14.3


## Importing necessary packages

In [13]:
# Libraries
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import os

# Statsmodels
import statsmodels.api as sm

# Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error


# Checking the src directory

In [14]:
CURRENT_DIRECTORY =  os.getcwd()
SRC_DIRECTORY = Path(CURRENT_DIRECTORY).parents[1]
print(SRC_DIRECTORY)

/Users/axlee/Desktop/Singhealth/AED-OHCA/src


# Importing datasets

In [15]:
BASE_DATASET_PATH = Path(SRC_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

/Users/axlee/Desktop/Singhealth/AED-OHCA/datasets


In [16]:
read_from_filepath = Path(BASE_DATASET_PATH / f"singapore_data/cleaned_data/ohca_data/ohca_binary_complete.csv")
final_model_df = pd.read_csv(read_from_filepath)
final_model_df = final_model_df.drop(columns=['Unnamed: 0'])

# Checking columns of the dataset

In [17]:
final_model_df.columns.tolist()

['pln_area_n',
 'subzone_n',
 'year',
 'month',
 'incident_count',
 'ohca_binary',
 'above_60_proportion',
 'male_chinese_proportion',
 'female_chinese_proportion',
 'male_malays_proportion',
 'female_malays_proportion',
 'male_indians_proportion',
 'female_indians_proportion',
 'male_others_proportion',
 'female_others_proportion',
 'business_1_encoding',
 'business_2_encoding',
 'business_park_encoding',
 'school_encoding',
 'airport',
 'is_residential']

## Creating predictor columns


In [18]:
predictor_cols = [
       'year', 'month',
       'above_60_proportion', 'male_chinese_proportion',
       'female_chinese_proportion', 'male_malays_proportion',
       'female_malays_proportion', 'male_indians_proportion',
       'female_indians_proportion', 'male_others_proportion',
       'female_others_proportion', 'business_1_encoding',
       'business_2_encoding', 'business_park_encoding', 'school_encoding',
       'airport', 'is_residential'
]

# Keep a reference dataframe for testing/comparison
comparison_df = final_model_df[['pln_area_n', 'subzone_n', 'year', 'month', 'ohca_binary']].copy()
comparison_df

,pln_area_n,subzone_n,year,month,ohca_binary
0,ang mo kio,ang mo kio town centre,2010,4,0
1,ang mo kio,ang mo kio town centre,2010,5,1
2,ang mo kio,ang mo kio town centre,2010,6,0
3,ang mo kio,ang mo kio town centre,2010,7,1
4,ang mo kio,ang mo kio town centre,2010,8,0
...,...,...,...,...,...
43564,western water catchment,murai,2021,8,0
43565,western water catchment,murai,2021,9,1
43566,western water catchment,murai,2021,10,1
43567,western water catchment,murai,2021,11,1


# Implementing Poisson_Distribution_Model

In [19]:
# Chronological Split
# Training: 2010 - 2019
train_df = final_model_df[final_model_df['year'] <= 2019]
X_train = train_df[predictor_cols].fillna(0)
y_train = train_df['incident_count']

# Validation: 2020
val_df = final_model_df[final_model_df['year'] == 2020]
X_val = val_df[predictor_cols].fillna(0)
y_val = val_df['incident_count']

# Testing: 2021
test_df = final_model_df[final_model_df['year'] == 2021]
X_test = test_df[predictor_cols].fillna(0)
y_test = test_df['incident_count']

In [20]:
# Add a constant (intercept) to the predictors
X_train_sm = sm.add_constant(X_train)

# Fit the basic Poisson model
poisson_model = sm.GLM(y_train, X_train_sm, family=sm.families.Poisson()).fit()
print(poisson_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:         incident_count   No. Observations:                36153
Model:                            GLM   Df Residuals:                    36135
Model Family:                 Poisson   Df Model:                           17
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -32588.
Date:                Wed, 04 Mar 2026   Deviance:                       36948.
Time:                        10:21:11   Pearson chi2:                 5.12e+04
No. Iterations:                     6   Pseudo R-squ. (CS):             0.3740
Covariance Type:            nonrobust                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
const                 

# Zero-inflated Poisson

In [21]:
# Define variables that predict whether a zone is an "always zero" state 
# (e.g., non-resi   dential zones, airports, parks)
inflation_cols = ['is_residential', 'airport'] 

# Added in constants column as intercept for inflation model
X_train_zip = sm.add_constant(X_train)
X_infl_zip = sm.add_constant(train_df[inflation_cols])

zip_model = sm.ZeroInflatedPoisson(endog=y_train, 
                                   exog=X_train_zip, 
                                   exog_infl=X_infl_zip, 
                                   inflation='logit').fit()

         Current function value: 0.900967
         Iterations: 35
         Function evaluations: 42
         Gradient evaluations: 42


/opt/miniconda3/envs/geospatial_env/lib/python3.14/site-packages/scipy/optimize/_optimize.py:1330: OptimizeWarning: Maximum number of iterations has been exceeded.
  res = _minimize_bfgs(f, x0, args, fprime, callback=callback, **opts)
/opt/miniconda3/envs/geospatial_env/lib/python3.14/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


# Metrics to check

In [22]:
# Example for test set
test_preds = poisson_model.predict(sm.add_constant(X_test, has_constant='add'))

print(f"RMSE: {np.sqrt(mean_squared_error(y_test, test_preds))}")
print(f"MAE: {mean_absolute_error(y_test, test_preds)}")

RMSE: 1.330129044678158
MAE: 0.8563071025410589
